In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
dataset_list = ["ARC_Challenge", "CommonSenseQA", "MMLU", "OpenBookQA"
                ]
model_list = [
    "EleutherAI/pythia-410m",
    "EleutherAI/pythia-1b",
    "EleutherAI/pythia-1.4b",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B",
    "Qwen/Qwen1.5-4B",
    "Qwen/Qwen1.5-0.5B",
    "Qwen/Qwen1.5-1.8B",
    "openai-community/gpt2",
    "openai-community/gpt2-large",
    "openai-community/gpt2-medium",
  ]

In [52]:
def get_mean_std(data): 
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)
    return mean, std

In [53]:
def get_data_list(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        results = [json.loads(line) for line in f]

    group_base = ["prompt_1", "prompt_2", "prompt_3"]
    group_first = ["prompt_4", "prompt_5", "prompt_6"]
    group_latter = ["prompt_7", "prompt_8", "prompt_9"]
    prompt_1_list = np.stack([r["prompt_key0"] for r in results], axis=0)  
    prompt_2_list = np.stack([r["prompt_key1"] for r in results], axis=0) 

    all_grad_list = np.stack([r["grads_norm_list"] for r in results], axis=0)              # shape: (num_samples, dim)
    all_delta_z_list = np.stack([r["delta_z_norm_list"] for r in results], axis=0)
    
    delta_log_prob_norm_list = [r["delta_log_prob_norm"] for r in results]

    first_grad_list = []
    first_delta_z_list = []
    first_delta_log_prob_norm_list = []
    latter_grad_list = []
    latter_delta_z_list = []
    latter_delta_log_prob_norm_list = []

    for p1, p2, grad, delta_z, delta_log_prob in zip(prompt_1_list, prompt_2_list, all_grad_list, all_delta_z_list, delta_log_prob_norm_list):
        if p1 == p2:
            continue
        if p1 in group_base and p2 in group_first:
            first_grad_list.append(grad)
            first_delta_z_list.append(delta_z)
            first_delta_log_prob_norm_list.append(delta_log_prob)
        elif p1 in group_base and p2 in group_latter:
            latter_grad_list.append(grad)
            latter_delta_z_list.append(delta_z)
            latter_delta_log_prob_norm_list.append(delta_log_prob)

    first_grad_list = np.array(first_grad_list)
    first_delta_z_list = np.array(first_delta_z_list)
    latter_grad_list = np.array(latter_grad_list)
    latter_delta_z_list = np.array(latter_delta_z_list)
    first_upper_bound_list = first_grad_list * first_delta_z_list
    latter_upper_bound_list = latter_grad_list * latter_delta_z_list

    first_grad_mean_list, first_grad_std_list = get_mean_std(first_grad_list)
    first_delta_z_mean_list, first_delta_z_std_list = get_mean_std(first_delta_z_list)
    latter_grad_mean_list, latter_grad_std_list = get_mean_std(latter_grad_list)
    latter_delta_z_mean_list, latter_delta_z_std_list = get_mean_std(latter_delta_z_list)
    first_upper_bound_mean_list, first_upper_bound_std_list = get_mean_std(first_upper_bound_list)
    latter_upper_bound_mean_list, latter_upper_bound_std_list = get_mean_std(latter_upper_bound_list)

    first_result_dict = {
        "grad_mean_list": first_grad_mean_list,
        "grad_std_list": first_grad_std_list,
        "delta_z_mean_list": first_delta_z_mean_list,
        "delta_z_std_list": first_delta_z_std_list,
        "upper_bound_mean_list": first_upper_bound_mean_list,
        "upper_bound_std_list": first_upper_bound_std_list,
        "delta_log_prob_norm_list": first_delta_log_prob_norm_list,
    }

    latter_result_dict = {
        "grad_mean_list": latter_grad_mean_list,
        "grad_std_list": latter_grad_std_list,
        "delta_z_mean_list": latter_delta_z_mean_list,
        "delta_z_std_list": latter_delta_z_std_list,
        "upper_bound_mean_list": latter_upper_bound_mean_list,
        "upper_bound_std_list": latter_upper_bound_std_list,
        "delta_log_prob_norm_list": latter_delta_log_prob_norm_list,
    }
    return first_result_dict, latter_result_dict

In [ ]:
def plot_line(first_delta_z_mean_list, 
              first_delta_z_std_list, 
              latter_delta_z_mean_list, 
              latter_delta_z_std_list,
              first_color, 
              latter_color, 
              dataset, 
              model_name_or_path, 
              label):
    plt.figure(figsize=(2.5, 2.5))

    x = np.arange(len(first_delta_z_mean_list))
    xticks = [str(i) for i in range(len(x))]
    step = max(1, len(x) // 4)

    first_mean = np.array(first_delta_z_mean_list)
    first_std = np.array(first_delta_z_std_list)
    first_lower = first_mean - first_std
    first_upper = first_mean + first_std

    latter_mean = np.array(latter_delta_z_mean_list)
    latter_std = np.array(latter_delta_z_std_list)
    latter_lower = latter_mean - latter_std
    latter_upper = latter_mean + latter_std

    plt.plot(
        x,
        first_delta_z_mean_list,
        label=r"First",
        color=first_color,
        linewidth=2
    )
    plt.plot(
        x,
        latter_delta_z_mean_list,
        label=r"Latter",
        color=latter_color,
        linewidth=2
    )
    plt.fill_between(
        x,
        first_lower,
        first_upper,
        color=first_color,
        alpha=0.1,
        linewidth=0
    )
    plt.fill_between(
        x,
        latter_lower,
        latter_upper,
        color=latter_color,
        alpha=0.1,
        linewidth=0
    )

    plt.xticks(x[::step], xticks[::step])

    plt.xlabel("Number of layers")
    plt.title("First vs. Latter")
    plt.legend()
    plt.grid(False)
    plt.tight_layout()
    
    save_path = f"../../results/figure_results/how_first_latter/{model_name_or_path}/{dataset}_{label}.pdf"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path)
    plt.close()


In [55]:
def plot_grads_and_deltaz(dataset, model_name_or_path):
    file_path = f"../../results/data_results/misalignment/{model_name_or_path}/{dataset}_result.jsonl"
    first_result_dict, latter_result_dict = get_data_list(file_path)
    
    first_grad_mean_list = first_result_dict["grad_mean_list"]
    first_grad_std_list = first_result_dict["grad_std_list"]
    first_delta_z_mean_list = first_result_dict["delta_z_mean_list"]
    first_delta_z_std_list = first_result_dict["delta_z_std_list"]
    first_upper_bound_mean_list = first_result_dict["upper_bound_mean_list"]
    first_upper_bound_std_list = first_result_dict["upper_bound_std_list"]
    first_delta_log_prob_norm_list = first_result_dict["delta_log_prob_norm_list"]

    latter_grad_mean_list = latter_result_dict["grad_mean_list"]
    latter_grad_std_list = latter_result_dict["grad_std_list"]
    latter_delta_z_mean_list = latter_result_dict["delta_z_mean_list"]
    latter_delta_z_std_list = latter_result_dict["delta_z_std_list"]
    latter_upper_bound_mean_list = latter_result_dict["upper_bound_mean_list"]
    latter_upper_bound_std_list = latter_result_dict["upper_bound_std_list"]
    latter_delta_log_prob_norm_list = latter_result_dict["delta_log_prob_norm_list"]

    first_mean_delta_log_prob_norm = np.mean(first_delta_log_prob_norm_list)
    latter_mean_delta_log_prob_norm = np.mean(latter_delta_log_prob_norm_list)

    first_color = "#0D4C6D"
    latter_color = "#FEB705"
    plot_line(first_delta_z_mean_list,
              first_delta_z_std_list, 
              latter_delta_z_mean_list, 
              latter_delta_z_std_list,
              first_color, 
              latter_color, 
              dataset, 
              model_name_or_path, 
              "first_latter")
   

In [56]:
for dataset in dataset_list:
    for model_name_or_path in model_list:
        plot_grads_and_deltaz(dataset, model_name_or_path)
